# 🎙️ Reconocimiento de Voz: Luz Inteligente

---

### **Proyecto Escolar — Minería de Datos**

| Campo | Detalle |
|---|---|
| **Proyecto** | Reconocimiento de Voz: Luz Inteligente |
| **Objetivo** | Clasificar comandos de voz *"encender luz"* y *"apagar luz"* mediante redes neuronales |

#### 👥 Integrantes
| # | Nombre |
|---|--------|
| 1 | Raziel Alejandro |
| 2 | Aaron González |
| 3 | Jordan Medina Ortiz |
| 4 | Lizeth Infante |
| 5 | Óscar |

| **Grupo** | *(Escribir grupo aquí)* |
|---|---|
| **Fecha de entrega** | 25 de mayo de 2026 |

---

## 📑 Índice

1. **Introducción Teórica**
   - 1.1 Procesamiento de Audio Digital
   - 1.2 Coeficientes Cepstrales en Frecuencias de Mel (MFCC)
   - 1.3 Redes Neuronales Densas (Fully Connected)
2. **Fase 1 — Preparación del Entorno**
   - Instalación de dependencias
   - Importación de librerías
3. **Fase 2 — Adquisición y Procesamiento de Datos**
   - Montaje de Google Drive
   - Función robusta de extracción de MFCC
   - Carga y etiquetado del dataset
   - Exploración y verificación de datos
4. **Fase 3 — Desarrollo del Modelo y Optimización (Tuning)**
   - Función `build_model` dinámica
   - Búsqueda de hiperparámetros
   - Selección del mejor modelo
   - Gráficas de Loss y Accuracy
5. **Fase 4 — Evaluación y Resultados**
   - Evaluación sobre conjunto de prueba
   - Lógica de predicción (Luz Encendida / Luz Apagada)
   - 🎤 Prueba Ciega con audio externo
6. **Sección de Redacción del Equipo**
   - Justificación de la Arquitectura
   - Ajustes Realizados
   - Conclusiones

---

# 1. Introducción Teórica

## 1.1 Procesamiento de Audio Digital

El **procesamiento de audio digital** consiste en convertir señales de sonido analógicas en representaciones numéricas que una computadora pueda analizar. Cuando grabamos un audio, el micrófono captura variaciones de presión en el aire y las convierte en una **señal digital** muestreada a una frecuencia determinada (por ejemplo, 22,050 Hz significa 22,050 muestras por segundo).

Para tareas de **reconocimiento de voz**, no trabajamos directamente con la forma de onda cruda (*waveform*), sino que extraemos **características espectrales** que capturan la información relevante del habla y descartan el ruido irrelevante. Esto reduce drásticamente la dimensionalidad del problema y mejora la capacidad del modelo para generalizar.

Las bibliotecas como `librosa` permiten cargar, transformar y analizar señales de audio en Python de forma eficiente, soportando múltiples formatos (WAV, MP3, OGG, FLAC, M4A) a través de backends como `soundfile` y `ffmpeg`.

## 1.2 Coeficientes Cepstrales en Frecuencias de Mel (MFCC)

Los **MFCC** (*Mel-Frequency Cepstral Coefficients*) son el estándar de facto para representar características de audio en tareas de reconocimiento de voz. El proceso de extracción sigue estos pasos:

1. **Ventaneo**: La señal se divide en ventanas cortas (~25 ms) con solapamiento.
2. **FFT**: Se calcula la Transformada Rápida de Fourier para obtener el espectro de frecuencias.
3. **Banco de filtros Mel**: Se aplica un banco de filtros triangulares en la escala Mel, que modela cómo el oído humano percibe las frecuencias (mayor resolución en frecuencias bajas).
4. **Logaritmo**: Se toma el logaritmo de las energías para comprimir el rango dinámico.
5. **DCT**: Se aplica la Transformada Discreta del Coseno para obtener los coeficientes finales.

El resultado es una matriz de forma `(n_mfcc, n_frames)` donde `n_mfcc` es el número de coeficientes (típicamente 13-40) y `n_frames` depende de la duración del audio.

## 1.3 Redes Neuronales Densas (*Fully Connected*)

Una **red neuronal densa** es la arquitectura más fundamental del aprendizaje profundo. Cada neurona de una capa está conectada con **todas** las neuronas de la capa siguiente. Para nuestro problema de clasificación binaria:

- **Capa de entrada**: Recibe el vector aplanado de MFCC.
- **Capas ocultas**: Aplican transformaciones no lineales (activación **ReLU**) para aprender patrones complejos.
- **Capa de salida**: Una sola neurona con activación **sigmoid** que produce una probabilidad entre 0 y 1.
  - `> 0.5` → "Encender luz"
  - `≤ 0.5` → "Apagar luz"

La función de pérdida **binary crossentropy** mide qué tan lejos están las predicciones de las etiquetas reales, y el optimizador **Adam** ajusta los pesos de la red para minimizar esta pérdida.

---

# 2. Fase 1 — Preparación del Entorno

En esta fase instalamos todas las dependencias necesarias e importamos las librerías que usaremos a lo largo del proyecto.

In [ ]:
# =============================================================================
# CELDA 1: Instalación de dependencias del sistema y paquetes de Python
# =============================================================================
# ffmpeg es necesario para decodificar formatos de audio como MP3, M4A, OGG, etc.
# pydub es una interfaz de alto nivel para manipulación de audio (usa ffmpeg internamente)
# librosa es la librería principal para análisis de audio y extracción de características
# soundfile es un backend alternativo de lectura/escritura de audio

# --- Instalar ffmpeg a nivel del sistema operativo (Linux/Colab) ---
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("✅ ffmpeg instalado correctamente.")

# --- Instalar paquetes de Python ---
!pip install -q librosa soundfile pydub
print("✅ librosa, soundfile y pydub instalados correctamente.")

In [ ]:
# =============================================================================
# CELDA 2: Importación de todas las librerías necesarias
# =============================================================================

# --- Procesamiento de audio ---
import librosa                          # Análisis y extracción de características de audio
import librosa.display                   # Visualización de espectrogramas y MFCCs
import soundfile as sf                   # Backend de lectura de audio (WAV, FLAC, OGG)

# --- Cálculo numérico ---
import numpy as np                       # Arrays y operaciones numéricas

# --- Visualización ---
import matplotlib.pyplot as plt          # Gráficas de entrenamiento y resultados
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)   # Tamaño por defecto de figuras
matplotlib.rcParams['figure.dpi'] = 100            # Resolución de figuras

# --- Machine Learning / Deep Learning ---
import tensorflow as tf                  # Framework de deep learning
from tensorflow import keras             # API de alto nivel para construir modelos
from tensorflow.keras.models import Sequential       # Modelo secuencial
from tensorflow.keras.layers import (
    Dense,       # Capa completamente conectada
    Flatten,     # Aplanar entrada multidimensional a 1D
    Dropout,     # Regularización para prevenir overfitting
    BatchNormalization,  # Normalización por lotes
    Input        # Capa de entrada explícita
)
from tensorflow.keras.callbacks import EarlyStopping  # Detención temprana

# --- Preprocesamiento y evaluación ---
from sklearn.model_selection import train_test_split  # División train/test
from sklearn.preprocessing import MinMaxScaler        # Normalización Min-Max
from sklearn.metrics import (
    classification_report,   # Reporte de precisión, recall, f1-score
    confusion_matrix,        # Matriz de confusión
    ConfusionMatrixDisplay   # Visualización de la matriz de confusión
)

# --- Utilidades del sistema ---
import os                                # Operaciones del sistema de archivos
import glob                              # Búsqueda de archivos por patrón
import warnings                          # Control de advertencias
warnings.filterwarnings('ignore')        # Suprimir warnings no críticos

# --- Verificar versiones ---
print("=" * 60)
print("  🔧 ENTORNO CONFIGURADO CORRECTAMENTE")
print("=" * 60)
print(f"  TensorFlow  : {tf.__version__}")
print(f"  Keras       : {keras.__version__}")
print(f"  Librosa     : {librosa.__version__}")
print(f"  NumPy       : {np.__version__}")
print(f"  GPU disp.   : {tf.config.list_physical_devices('GPU')}")
print("=" * 60)

# --- Fijar semilla para reproducibilidad ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"\n🎲 Semilla fijada en {SEED} para reproducibilidad.")

---

# 3. Fase 2 — Adquisición y Procesamiento de Datos

En esta fase:
1. Montamos Google Drive para acceder al dataset.
2. Definimos una función **robusta** de extracción de MFCC compatible con múltiples formatos.
3. Iteramos sobre las carpetas del dataset y construimos los arrays `X` (características) e `y` (etiquetas).

### 📁 Estructura esperada del dataset en Google Drive

```
Mi unidad/
  └── dataset_luz/
      ├── encender/
      │   ├── audio_001.wav
      │   ├── audio_002.mp3
      │   └── ...  (archivos en cualquier formato)
      └── apagar/
          ├── audio_001.wav
          ├── audio_002.ogg
          └── ...
```

> ⚠️ **Nota:** Modifica la variable `BASE_DIR` en la celda de carga si tu estructura de carpetas es diferente.

In [ ]:
# =============================================================================
# CELDA 3: Montar Google Drive
# =============================================================================
# Esto nos permite acceder a los archivos de audio almacenados en Drive.
# Se abrirá una ventana de autenticación la primera vez.

from google.colab import drive
drive.mount('/content/drive')

print("\n📂 Google Drive montado en /content/drive")

In [ ]:
# =============================================================================
# CELDA 4: Configuración global de parámetros de audio
# =============================================================================
# Centralizamos todos los hiperparámetros de procesamiento de audio aquí
# para facilitar la experimentación y mantener la consistencia.

# --- Parámetros de carga de audio ---
SAMPLE_RATE = 22050      # Frecuencia de muestreo en Hz (estándar para análisis de voz)
                         # 22,050 muestras por segundo es suficiente para capturar
                         # todas las frecuencias relevantes del habla humana (hasta ~11 kHz)

# --- Parámetros de extracción de MFCC ---
N_MFCC = 40              # Número de coeficientes MFCC a extraer
                         # 40 es un buen balance: captura suficiente detalle espectral
                         # sin ser excesivamente dimensional

N_FFT = 2048             # Tamaño de la ventana FFT (en muestras)
                         # ~93 ms a 22,050 Hz — ventana estándar para análisis de voz

HOP_LENGTH = 512         # Desplazamiento entre ventanas consecutivas (en muestras)
                         # ~23 ms — determina la resolución temporal

MAX_PAD_LEN = 100        # Número máximo de frames temporales
                         # Todos los audios se ajustarán a exactamente esta longitud
                         # 100 frames × 23 ms ≈ 2.3 segundos de audio
                         # Suficiente para comandos cortos como "encender luz"

# --- Formatos de audio soportados ---
FORMATOS_SOPORTADOS = ('.wav', '.mp3', '.ogg', '.flac', '.m4a', '.aac', '.wma', '.opus')

print("⚙️  Parámetros de procesamiento configurados:")
print(f"   Sample Rate     : {SAMPLE_RATE} Hz")
print(f"   N_MFCC          : {N_MFCC} coeficientes")
print(f"   N_FFT           : {N_FFT} muestras")
print(f"   Hop Length      : {HOP_LENGTH} muestras")
print(f"   Max Pad Length  : {MAX_PAD_LEN} frames")
print(f"   Vector final    : {N_MFCC} × {MAX_PAD_LEN} = {N_MFCC * MAX_PAD_LEN} características")
print(f"   Formatos        : {', '.join(FORMATOS_SOPORTADOS)}")

In [ ]:
# =============================================================================
# CELDA 5: Función ROBUSTA de extracción de características MFCC
# =============================================================================
# Esta función es el corazón del preprocesamiento. Está diseñada para:
#   1. Leer CUALQUIER formato de audio soportado por librosa + ffmpeg
#   2. Convertir a mono si el audio es estéreo
#   3. Re-muestrear a la frecuencia estándar (22,050 Hz)
#   4. Extraer los coeficientes MFCC
#   5. Aplicar padding (rellenar con ceros) o truncating (recortar)
#      para garantizar dimensiones uniformes
#   6. Normalizar los valores al rango [0, 1]

def extraer_mfcc(ruta_audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC, n_fft=N_FFT,
                 hop_length=HOP_LENGTH, max_pad_len=MAX_PAD_LEN):
    """
    Extrae los MFCC de un archivo de audio de cualquier formato.

    Parámetros
    ----------
    ruta_audio : str
        Ruta completa al archivo de audio.
    sr : int
        Frecuencia de muestreo objetivo.
    n_mfcc : int
        Número de coeficientes MFCC.
    n_fft : int
        Tamaño de la ventana FFT.
    hop_length : int
        Salto entre ventanas.
    max_pad_len : int
        Longitud temporal fija (en frames) para padding/truncating.

    Retorna
    -------
    numpy.ndarray
        Matriz MFCC normalizada de forma (n_mfcc, max_pad_len).
        Retorna None si ocurre un error irrecuperable.
    """
    try:
        # -----------------------------------------------------------------
        # PASO 1: Cargar el audio
        # -----------------------------------------------------------------
        # librosa.load() maneja automáticamente:
        #   - Conversión de estéreo a mono (mono=True por defecto)
        #   - Re-muestreo a la frecuencia 'sr' especificada
        #   - Decodificación de múltiples formatos via soundfile o audioread/ffmpeg
        señal, _ = librosa.load(ruta_audio, sr=sr, mono=True)

        # Verificar que la señal no esté vacía
        if len(señal) == 0:
            print(f"   ⚠️  Audio vacío: {os.path.basename(ruta_audio)}")
            return None

        # -----------------------------------------------------------------
        # PASO 2: Extraer los MFCC
        # -----------------------------------------------------------------
        # La función librosa.feature.mfcc() realiza internamente:
        #   1. Calcula el espectrograma de Mel
        #   2. Aplica logaritmo a las energías
        #   3. Aplica la DCT para obtener los coeficientes
        # Resultado: matriz de forma (n_mfcc, n_frames)
        mfccs = librosa.feature.mfcc(
            y=señal,
            sr=sr,
            n_mfcc=n_mfcc,
            n_fft=n_fft,
            hop_length=hop_length
        )

        # -----------------------------------------------------------------
        # PASO 3: Padding o Truncating
        # -----------------------------------------------------------------
        # Necesitamos que TODOS los audios tengan exactamente el mismo
        # número de frames temporales para poder alimentar la red neuronal.
        n_frames_actual = mfccs.shape[1]

        if n_frames_actual < max_pad_len:
            # Si el audio es más corto → PADDING (rellenar con ceros a la derecha)
            pad_width = max_pad_len - n_frames_actual
            mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        elif n_frames_actual > max_pad_len:
            # Si el audio es más largo → TRUNCATING (recortar al tamaño máximo)
            mfccs = mfccs[:, :max_pad_len]

        # -----------------------------------------------------------------
        # PASO 4: Normalización Min-Max al rango [0, 1]
        # -----------------------------------------------------------------
        # Normalizamos cada matriz MFCC individualmente para que todos los
        # valores estén en el mismo rango, facilitando el entrenamiento.
        mfcc_min = mfccs.min()
        mfcc_max = mfccs.max()

        if mfcc_max - mfcc_min > 0:
            # Normalización estándar: (x - min) / (max - min)
            mfccs = (mfccs - mfcc_min) / (mfcc_max - mfcc_min)
        else:
            # Caso especial: señal constante (silencio total)
            mfccs = np.zeros_like(mfccs)

        return mfccs

    except Exception as e:
        # Si ocurre cualquier error, lo reportamos pero no detenemos el proceso
        print(f"   ❌ Error procesando '{os.path.basename(ruta_audio)}': {e}")
        return None


# --- Prueba rápida de la función ---
print("✅ Función extraer_mfcc() definida correctamente.")
print(f"   Forma esperada de salida: ({N_MFCC}, {MAX_PAD_LEN})")

In [ ]:
# =============================================================================
# CELDA 6: Cargar y procesar TODOS los audios del dataset
# =============================================================================
# Iteramos sobre las carpetas del dataset, procesamos cada audio y
# construimos los arrays X (características) e y (etiquetas).

# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚠️  MODIFICA ESTA RUTA si tu dataset está en otra ubicación   ║
# ╚══════════════════════════════════════════════════════════════════╝
BASE_DIR = '/content/drive/MyDrive/dataset_luz/'

# Mapeo de carpetas a etiquetas numéricas
# Etiqueta 1 = "encender luz" (predicción positiva)
# Etiqueta 0 = "apagar luz"   (predicción negativa)
CARPETAS_ETIQUETAS = {
    'encender': 1,
    'apagar': 0
}

# Listas para acumular los datos procesados
lista_mfcc = []      # Almacenará las matrices MFCC de cada audio
lista_etiquetas = [] # Almacenará la etiqueta correspondiente (0 o 1)

# Contadores para el reporte final
total_procesados = 0
total_errores = 0
conteo_por_clase = {'encender': 0, 'apagar': 0}

print("=" * 60)
print("  📂 PROCESANDO DATASET DE AUDIO")
print("=" * 60)

for carpeta, etiqueta in CARPETAS_ETIQUETAS.items():
    ruta_carpeta = os.path.join(BASE_DIR, carpeta)

    # Verificar que la carpeta existe
    if not os.path.isdir(ruta_carpeta):
        print(f"\n❌ ERROR: No se encontró la carpeta '{ruta_carpeta}'")
        print(f"   Verifica que la ruta BASE_DIR sea correcta.")
        continue

    # Obtener TODOS los archivos de audio de la carpeta
    # Filtramos por las extensiones de formatos soportados
    archivos = [
        f for f in os.listdir(ruta_carpeta)
        if f.lower().endswith(FORMATOS_SOPORTADOS)
    ]
    archivos.sort()  # Ordenar alfabéticamente para consistencia

    print(f"\n📁 Carpeta: '{carpeta}/' — {len(archivos)} archivos encontrados")
    print(f"   Etiqueta asignada: {etiqueta} ({'Encender' if etiqueta == 1 else 'Apagar'})")
    print(f"   {'—' * 50}")

    for i, archivo in enumerate(archivos, 1):
        ruta_completa = os.path.join(ruta_carpeta, archivo)

        # Extraer MFCC usando nuestra función robusta
        mfcc = extraer_mfcc(ruta_completa)

        if mfcc is not None:
            lista_mfcc.append(mfcc)
            lista_etiquetas.append(etiqueta)
            total_procesados += 1
            conteo_por_clase[carpeta] += 1

            # Imprimir progreso cada 10 archivos
            if i % 10 == 0 or i == len(archivos):
                print(f"   ✅ Procesados: {i}/{len(archivos)}")
        else:
            total_errores += 1

# =============================================================================
# Convertir listas a arrays de NumPy
# =============================================================================
X = np.array(lista_mfcc)          # Shape: (n_audios, N_MFCC, MAX_PAD_LEN)
y = np.array(lista_etiquetas)     # Shape: (n_audios,)

# --- Reporte final ---
print(f"\n{'=' * 60}")
print(f"  📊 RESUMEN DEL PROCESAMIENTO")
print(f"{'=' * 60}")
print(f"  Total procesados : {total_procesados} audios")
print(f"  Total errores    : {total_errores} audios")
print(f"  Clase 'encender' : {conteo_por_clase['encender']} audios (etiqueta=1)")
print(f"  Clase 'apagar'   : {conteo_por_clase['apagar']} audios (etiqueta=0)")
print(f"  Forma de X       : {X.shape}")
print(f"  Forma de y       : {y.shape}")
print(f"{'=' * 60}")

In [ ]:
# =============================================================================
# CELDA 7: Exploración y verificación visual de los datos
# =============================================================================
# Visualizamos algunos MFCCs para verificar que el procesamiento fue correcto
# y entender cómo se ven las características de cada clase.

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle('🔍 Muestras de MFCC por Clase', fontsize=16, fontweight='bold')

# Seleccionar 3 muestras aleatorias de cada clase
indices_encender = np.where(y == 1)[0]
indices_apagar = np.where(y == 0)[0]

# Verificar que hay suficientes muestras
n_muestras = min(3, len(indices_encender), len(indices_apagar))

for i in range(n_muestras):
    # Fila superior: "Encender luz"
    ax_enc = axes[0, i]
    img_enc = ax_enc.imshow(X[indices_encender[i]], aspect='auto', origin='lower', cmap='magma')
    ax_enc.set_title(f'Encender #{i+1}', fontsize=12)
    ax_enc.set_ylabel('Coeficiente MFCC' if i == 0 else '')
    ax_enc.set_xlabel('Frame temporal')
    plt.colorbar(img_enc, ax=ax_enc, fraction=0.046)

    # Fila inferior: "Apagar luz"
    ax_apa = axes[1, i]
    img_apa = ax_apa.imshow(X[indices_apagar[i]], aspect='auto', origin='lower', cmap='magma')
    ax_apa.set_title(f'Apagar #{i+1}', fontsize=12)
    ax_apa.set_ylabel('Coeficiente MFCC' if i == 0 else '')
    ax_apa.set_xlabel('Frame temporal')
    plt.colorbar(img_apa, ax=ax_apa, fraction=0.046)

plt.tight_layout()
plt.show()

# --- Distribución de clases ---
fig2, ax2 = plt.subplots(figsize=(6, 4))
clases, conteos = np.unique(y, return_counts=True)
colores = ['#FF6B6B', '#4ECDC4']
barras = ax2.bar(['Apagar (0)', 'Encender (1)'], conteos, color=colores, edgecolor='black')
ax2.set_title('📊 Distribución de Clases en el Dataset', fontsize=14, fontweight='bold')
ax2.set_ylabel('Número de audios')
for barra, conteo in zip(barras, conteos):
    ax2.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.5,
             str(conteo), ha='center', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

# 4. Fase 3 — Desarrollo del Modelo y Optimización (Tuning)

En esta fase:
1. Creamos una función que construye modelos de forma **dinámica** según los hiperparámetros.
2. Probamos **9 combinaciones** de arquitecturas (1, 2 o 3 capas ocultas × 32, 64 o 128 neuronas).
3. Seleccionamos el **mejor modelo** basándonos en la accuracy de validación.
4. Graficamos las curvas de entrenamiento del modelo ganador.

In [ ]:
# =============================================================================
# CELDA 8: División del dataset en entrenamiento y prueba
# =============================================================================
# Dividimos los datos ANTES de entrenar cualquier modelo para evitar
# fugas de información (data leakage).

# Aplanar los MFCC de (n_audios, N_MFCC, MAX_PAD_LEN) a (n_audios, N_MFCC*MAX_PAD_LEN)
# Esto es necesario porque usaremos capas Dense que esperan entrada 1D
X_flat = X.reshape(X.shape[0], -1)

print(f"Forma de X original : {X.shape}")
print(f"Forma de X aplanado : {X_flat.shape}")
print(f"Cada audio tiene {X_flat.shape[1]} características (features)\n")

# División: 80% entrenamiento, 20% prueba
# stratify=y asegura que ambas clases estén proporcionalmente representadas
X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y           # Mantener la proporción de clases en ambos conjuntos
)

print(f"🔹 Conjunto de ENTRENAMIENTO: {X_train.shape[0]} audios")
print(f"   Encender: {np.sum(y_train == 1)} | Apagar: {np.sum(y_train == 0)}")
print(f"\n🔸 Conjunto de PRUEBA: {X_test.shape[0]} audios")
print(f"   Encender: {np.sum(y_test == 1)} | Apagar: {np.sum(y_test == 0)}")

In [ ]:
# =============================================================================
# CELDA 9: Función build_model — Construcción dinámica del modelo
# =============================================================================
# Esta función permite construir redes neuronales con diferente número de
# capas ocultas y neuronas, facilitando la búsqueda de hiperparámetros.

def build_model(input_dim, hidden_layers=2, neurons=64, dropout_rate=0.3):
    """
    Construye un modelo Keras Sequential para clasificación binaria.

    Parámetros
    ----------
    input_dim : int
        Dimensión del vector de entrada (número de características).
    hidden_layers : int
        Número de capas ocultas Dense (1, 2 o 3).
    neurons : int
        Número de neuronas por capa oculta (32, 64 o 128).
    dropout_rate : float
        Fracción de neuronas a desactivar aleatoriamente (regularización).

    Retorna
    -------
    keras.Sequential
        Modelo compilado listo para entrenar.
    """
    model = Sequential(name=f'modelo_{hidden_layers}capas_{neurons}neuronas')

    # --- Capa de entrada ---
    # Especificamos explícitamente la forma de entrada
    model.add(Input(shape=(input_dim,)))

    # --- Capas ocultas dinámicas ---
    for i in range(hidden_layers):
        # Capa Dense con activación ReLU
        # ReLU (Rectified Linear Unit): f(x) = max(0, x)
        # Es la activación estándar para capas ocultas por su eficiencia
        model.add(Dense(
            units=neurons,
            activation='relu',
            name=f'dense_oculta_{i+1}'
        ))

        # BatchNormalization: normaliza las activaciones de la capa anterior
        # Esto acelera el entrenamiento y permite usar learning rates más altos
        model.add(BatchNormalization(name=f'batch_norm_{i+1}'))

        # Dropout: desactiva aleatoriamente un % de neuronas durante entrenamiento
        # Esto previene el overfitting al obligar a la red a no depender
        # de neuronas específicas
        model.add(Dropout(dropout_rate, name=f'dropout_{i+1}'))

    # --- Capa de salida ---
    # Una sola neurona con activación sigmoid para clasificación binaria
    # Sigmoid: f(x) = 1 / (1 + e^(-x)) → salida en el rango (0, 1)
    # Interpretación: probabilidad de que el audio sea "encender luz"
    model.add(Dense(1, activation='sigmoid', name='salida'))

    # --- Compilación ---
    # binary_crossentropy: función de pérdida estándar para clasificación binaria
    # Adam: optimizador adaptativo que ajusta el learning rate automáticamente
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


# --- Ejemplo: mostrar la arquitectura de un modelo de prueba ---
print("📐 Ejemplo de arquitectura (2 capas, 64 neuronas):")
modelo_ejemplo = build_model(input_dim=X_train.shape[1], hidden_layers=2, neurons=64)
modelo_ejemplo.summary()

In [ ]:
# =============================================================================
# CELDA 10: Búsqueda de hiperparámetros (Tuning)
# =============================================================================
# Probamos TODAS las combinaciones de:
#   - Número de capas ocultas: 1, 2, 3
#   - Número de neuronas por capa: 32, 64, 128
# Total: 3 × 3 = 9 configuraciones

# Hiperparámetros a explorar
OPCIONES_CAPAS = [1, 2, 3]
OPCIONES_NEURONAS = [32, 64, 128]

# Configuración de entrenamiento
EPOCHS = 100          # Número máximo de épocas (EarlyStopping detendrá antes)
BATCH_SIZE = 16       # Tamaño del lote (ajustado para datasets pequeños)
PATIENCE = 15         # Épocas sin mejora antes de detener el entrenamiento
VALIDATION_SPLIT = 0.2  # 20% de los datos de entrenamiento para validación

# Almacenar resultados de cada configuración
resultados = []       # Lista de diccionarios con métricas de cada modelo
historiales = {}      # Historiales de entrenamiento indexados por nombre

print("=" * 70)
print("  🔄 BÚSQUEDA DE HIPERPARÁMETROS")
print(f"  Combinaciones a probar: {len(OPCIONES_CAPAS) * len(OPCIONES_NEURONAS)}")
print(f"  Epochs máx: {EPOCHS} | Batch: {BATCH_SIZE} | Patience: {PATIENCE}")
print("=" * 70)

for n_capas in OPCIONES_CAPAS:
    for n_neuronas in OPCIONES_NEURONAS:
        nombre = f"{n_capas}cap_{n_neuronas}neu"
        print(f"\n{'─' * 50}")
        print(f"🧪 Entrenando: {n_capas} capa(s) × {n_neuronas} neuronas")
        print(f"{'─' * 50}")

        # Construir el modelo con los hiperparámetros actuales
        modelo = build_model(
            input_dim=X_train.shape[1],
            hidden_layers=n_capas,
            neurons=n_neuronas,
            dropout_rate=0.3
        )

        # Callback de EarlyStopping:
        # Monitorea la pérdida de validación (val_loss)
        # Si no mejora después de 'patience' épocas, detiene el entrenamiento
        # restore_best_weights: restaura los pesos de la mejor época
        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=0
        )

        # Entrenar el modelo
        historial = modelo.fit(
            X_train, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_split=VALIDATION_SPLIT,
            callbacks=[early_stop],
            verbose=0  # Silenciar salida para mantener limpio el log
        )

        # Evaluar sobre el conjunto de PRUEBA (datos nunca vistos)
        loss_test, acc_test = modelo.evaluate(X_test, y_test, verbose=0)

        # Obtener la mejor accuracy de validación durante el entrenamiento
        mejor_val_acc = max(historial.history['val_accuracy'])
        epocas_entrenadas = len(historial.history['loss'])

        # Almacenar resultados
        resultados.append({
            'nombre': nombre,
            'capas': n_capas,
            'neuronas': n_neuronas,
            'val_accuracy': mejor_val_acc,
            'test_accuracy': acc_test,
            'test_loss': loss_test,
            'epocas': epocas_entrenadas,
            'modelo': modelo
        })
        historiales[nombre] = historial

        print(f"   📈 Val Accuracy : {mejor_val_acc:.4f}")
        print(f"   📊 Test Accuracy: {acc_test:.4f}")
        print(f"   📉 Test Loss    : {loss_test:.4f}")
        print(f"   ⏱️  Épocas       : {epocas_entrenadas}")

print(f"\n{'=' * 70}")
print("  ✅ BÚSQUEDA DE HIPERPARÁMETROS COMPLETADA")
print(f"{'=' * 70}")

In [ ]:
# =============================================================================
# CELDA 11: Tabla comparativa y selección del mejor modelo
# =============================================================================
# Mostramos todos los resultados ordenados por test_accuracy
# y seleccionamos el mejor modelo.

# Ordenar resultados por test_accuracy (de mayor a menor)
resultados_ordenados = sorted(resultados, key=lambda x: x['test_accuracy'], reverse=True)

print("\n" + "=" * 80)
print("  🏆 TABLA COMPARATIVA DE MODELOS (ordenados por Test Accuracy)")
print("=" * 80)
print(f"  {'#':<4} {'Configuración':<20} {'Val Acc':>10} {'Test Acc':>10} {'Test Loss':>10} {'Épocas':>8}")
print(f"  {'─'*4} {'─'*20} {'─'*10} {'─'*10} {'─'*10} {'─'*8}")

for i, r in enumerate(resultados_ordenados, 1):
    marca = " 🥇" if i == 1 else (" 🥈" if i == 2 else (" 🥉" if i == 3 else "   "))
    print(f"{marca}{i:<3} {r['nombre']:<20} {r['val_accuracy']:>10.4f} {r['test_accuracy']:>10.4f} {r['test_loss']:>10.4f} {r['epocas']:>8}")

# --- Seleccionar el MEJOR modelo ---
mejor = resultados_ordenados[0]
mejor_modelo = mejor['modelo']
mejor_nombre = mejor['nombre']
mejor_historial = historiales[mejor_nombre]

print(f"\n{'=' * 80}")
print(f"  🏆 MEJOR MODELO: {mejor_nombre}")
print(f"     Capas ocultas : {mejor['capas']}")
print(f"     Neuronas/capa : {mejor['neuronas']}")
print(f"     Test Accuracy : {mejor['test_accuracy']:.4f} ({mejor['test_accuracy']*100:.2f}%)")
print(f"     Test Loss     : {mejor['test_loss']:.4f}")
print(f"{'=' * 80}")

# Mostrar arquitectura del mejor modelo
print("\n📐 Arquitectura del mejor modelo:")
mejor_modelo.summary()

In [ ]:
# =============================================================================
# CELDA 12: Gráficas de Loss y Accuracy del MEJOR modelo
# =============================================================================
# Visualizamos las curvas de entrenamiento para diagnosticar:
#   - Overfitting: si val_loss sube mientras train_loss baja
#   - Underfitting: si ambas curvas son altas y planas
#   - Buen ajuste: si ambas curvas convergen a valores bajos

hist = mejor_historial.history

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'📈 Curvas de Entrenamiento — Mejor Modelo: {mejor_nombre}',
             fontsize=16, fontweight='bold')

# --- Gráfica de LOSS (Pérdida) ---
ax1.plot(hist['loss'], label='Train Loss', color='#3498DB', linewidth=2)
ax1.plot(hist['val_loss'], label='Val Loss', color='#E74C3C', linewidth=2, linestyle='--')
ax1.set_title('Función de Pérdida (Binary Crossentropy)', fontsize=13)
ax1.set_xlabel('Época', fontsize=12)
ax1.set_ylabel('Pérdida', fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_facecolor('#FAFAFA')

# Marcar la época con menor val_loss
mejor_epoca_loss = np.argmin(hist['val_loss'])
ax1.axvline(x=mejor_epoca_loss, color='gray', linestyle=':', alpha=0.7)
ax1.annotate(f'Mejor val_loss\nÉpoca {mejor_epoca_loss+1}',
             xy=(mejor_epoca_loss, hist['val_loss'][mejor_epoca_loss]),
             xytext=(mejor_epoca_loss+5, max(hist['val_loss'])*0.7),
             arrowprops=dict(arrowstyle='->', color='gray'),
             fontsize=10, color='gray')

# --- Gráfica de ACCURACY (Precisión) ---
ax2.plot(hist['accuracy'], label='Train Accuracy', color='#2ECC71', linewidth=2)
ax2.plot(hist['val_accuracy'], label='Val Accuracy', color='#F39C12', linewidth=2, linestyle='--')
ax2.set_title('Precisión (Accuracy)', fontsize=13)
ax2.set_xlabel('Época', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_facecolor('#FAFAFA')
ax2.set_ylim([0, 1.05])  # Accuracy va de 0 a 1

# Marcar la época con mejor val_accuracy
mejor_epoca_acc = np.argmax(hist['val_accuracy'])
ax2.axvline(x=mejor_epoca_acc, color='gray', linestyle=':', alpha=0.7)
ax2.annotate(f'Mejor val_acc\n{hist["val_accuracy"][mejor_epoca_acc]:.4f}',
             xy=(mejor_epoca_acc, hist['val_accuracy'][mejor_epoca_acc]),
             xytext=(mejor_epoca_acc+5, 0.5),
             arrowprops=dict(arrowstyle='->', color='gray'),
             fontsize=10, color='gray')

plt.tight_layout()
plt.show()

# --- Gráfica comparativa de todos los modelos ---
fig3, ax3 = plt.subplots(figsize=(12, 6))
nombres = [r['nombre'] for r in resultados_ordenados]
test_accs = [r['test_accuracy'] for r in resultados_ordenados]
val_accs = [r['val_accuracy'] for r in resultados_ordenados]

x_pos = np.arange(len(nombres))
width = 0.35

barras1 = ax3.bar(x_pos - width/2, val_accs, width, label='Val Accuracy',
                  color='#3498DB', edgecolor='black', alpha=0.85)
barras2 = ax3.bar(x_pos + width/2, test_accs, width, label='Test Accuracy',
                  color='#2ECC71', edgecolor='black', alpha=0.85)

ax3.set_title('🏆 Comparativa de Accuracy por Configuración', fontsize=14, fontweight='bold')
ax3.set_xlabel('Configuración del modelo', fontsize=12)
ax3.set_ylabel('Accuracy', fontsize=12)
ax3.set_xticks(x_pos)
ax3.set_xticklabels(nombres, rotation=45, ha='right')
ax3.legend(fontsize=11)
ax3.set_ylim([0, 1.1])
ax3.grid(axis='y', alpha=0.3)

# Añadir valores sobre las barras
for barra in barras1:
    ax3.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.01,
             f'{barra.get_height():.3f}', ha='center', fontsize=8)
for barra in barras2:
    ax3.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.01,
             f'{barra.get_height():.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

---

# 5. Fase 4 — Evaluación y Resultados

En esta fase evaluamos el mejor modelo de forma exhaustiva:
1. **Reporte de clasificación** con precisión, recall y F1-score.
2. **Matriz de confusión** visualizada.
3. **Lógica de predicción** con salida legible.
4. **Prueba ciega** con audio externo (voz del profesor).

In [ ]:
# =============================================================================
# CELDA 13: Evaluación detallada del mejor modelo
# =============================================================================
# Generamos predicciones sobre el conjunto de prueba y calculamos métricas.

# Obtener predicciones (probabilidades) del mejor modelo
y_pred_prob = mejor_modelo.predict(X_test, verbose=0).flatten()

# Convertir probabilidades a clases binarias (umbral = 0.5)
y_pred = (y_pred_prob > 0.5).astype(int)

# --- Reporte de clasificación ---
print("=" * 60)
print("  📋 REPORTE DE CLASIFICACIÓN")
print("=" * 60)
print(classification_report(
    y_test, y_pred,
    target_names=['Apagar (0)', 'Encender (1)'],
    digits=4
))

# --- Matriz de confusión ---
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Apagar', 'Encender']
)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('🔲 Matriz de Confusión — Mejor Modelo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Interpretación ---
print("\n📖 Interpretación de la Matriz de Confusión:")
print(f"   Verdaderos Negativos (Apagar correctos)  : {cm[0,0]}")
print(f"   Falsos Positivos (Apagar → Encender)     : {cm[0,1]}")
print(f"   Falsos Negativos (Encender → Apagar)     : {cm[1,0]}")
print(f"   Verdaderos Positivos (Encender correctos): {cm[1,1]}")

In [ ]:
# =============================================================================
# CELDA 14: Lógica de predicción — "Luz Encendida" / "Luz Apagada"
# =============================================================================
# Función que toma la salida del modelo y produce una decisión legible.

def predecir_comando(modelo, mfcc_features):
    """
    Realiza la predicción sobre un audio preprocesado y muestra el resultado.

    Parámetros
    ----------
    modelo : keras.Model
        Modelo entrenado.
    mfcc_features : numpy.ndarray
        Matriz MFCC del audio (ya preprocesada y normalizada).

    Retorna
    -------
    str
        Comando reconocido.
    """
    # Aplanar la matriz MFCC y añadir dimensión de batch
    entrada = mfcc_features.flatten().reshape(1, -1)

    # Obtener la probabilidad del modelo
    probabilidad = modelo.predict(entrada, verbose=0)[0][0]

    # Aplicar el umbral de decisión
    print("\n" + "═" * 50)
    print(f"  🎯 Probabilidad del modelo: {probabilidad:.4f}")
    print("═" * 50)

    if probabilidad > 0.5:
        comando = "ENCENDER LUZ"
        print(f"  💡 Predicción: Luz Encendida")
        print(f"  🟢 Comando reconocido: '{comando}'")
        print(f"  📊 Confianza: {probabilidad * 100:.1f}%")
    else:
        comando = "APAGAR LUZ"
        print(f"  🌑 Predicción: Luz Apagada")
        print(f"  🔴 Comando reconocido: '{comando}'")
        print(f"  📊 Confianza: {(1 - probabilidad) * 100:.1f}%")

    print("═" * 50)
    return comando


# --- Demostración con muestras del conjunto de prueba ---
print("🔎 Demostración con 5 muestras del conjunto de prueba:\n")
for i in range(min(5, len(X_test))):
    print(f"\n--- Muestra {i+1} (Etiqueta real: {'Encender' if y_test[i] == 1 else 'Apagar'}) ---")
    # Reconstruir la forma 2D para la función de predicción
    mfcc_2d = X_test[i].reshape(N_MFCC, MAX_PAD_LEN)
    predecir_comando(mejor_modelo, mfcc_2d)

---

## 🎤 Prueba Ciega — Audio Externo

En esta sección cargamos un audio **completamente nuevo** (por ejemplo, la voz del profesor) para probar la capacidad de generalización del modelo.

**Instrucciones:**
1. Sube el audio a Google Drive o directamente a Colab.
2. Copia la ruta del archivo.
3. Pégala en la variable `RUTA_AUDIO_PRUEBA` de la siguiente celda.
4. Ejecuta la celda.

> ⚠️ El audio puede estar en **cualquier formato**: WAV, MP3, OGG, FLAC, M4A, etc.

In [ ]:
# =============================================================================
# CELDA 15: PRUEBA CIEGA — Predicción con audio externo
# =============================================================================
# Esta celda permite cargar un audio completamente nuevo (fuera del dataset)
# y obtener la predicción del modelo.

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ⚠️  MODIFICA ESTA RUTA con el audio que deseas probar                 ║
# ║  Acepta cualquier formato: .wav, .mp3, .ogg, .flac, .m4a, etc.         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
RUTA_AUDIO_PRUEBA = '/content/drive/MyDrive/prueba_ciega/audio_profesor.wav'

# --- Verificar que el archivo existe ---
if not os.path.exists(RUTA_AUDIO_PRUEBA):
    print(f"❌ ERROR: No se encontró el archivo:")
    print(f"   {RUTA_AUDIO_PRUEBA}")
    print(f"\n💡 Opciones:")
    print(f"   1. Sube el audio a Google Drive y actualiza la ruta.")
    print(f"   2. Usa el panel de archivos de Colab (ícono de carpeta a la izquierda)")
    print(f"      para subir el archivo directamente a /content/")
else:
    print(f"📂 Archivo encontrado: {os.path.basename(RUTA_AUDIO_PRUEBA)}")
    print(f"   Tamaño: {os.path.getsize(RUTA_AUDIO_PRUEBA) / 1024:.1f} KB")
    print(f"   Formato: {os.path.splitext(RUTA_AUDIO_PRUEBA)[1]}")

    # --- Aplicar el MISMO preprocesamiento de la Fase 2 ---
    print("\n🔄 Aplicando preprocesamiento MFCC...")
    mfcc_prueba = extraer_mfcc(RUTA_AUDIO_PRUEBA)

    if mfcc_prueba is not None:
        print(f"   Forma del MFCC extraído: {mfcc_prueba.shape}")
        print(f"   Rango de valores: [{mfcc_prueba.min():.4f}, {mfcc_prueba.max():.4f}]")

        # --- Visualizar el MFCC del audio de prueba ---
        fig, ax = plt.subplots(figsize=(12, 4))
        img = ax.imshow(mfcc_prueba, aspect='auto', origin='lower', cmap='magma')
        ax.set_title(f'MFCC — Audio de Prueba Ciega: {os.path.basename(RUTA_AUDIO_PRUEBA)}',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Frame temporal')
        ax.set_ylabel('Coeficiente MFCC')
        plt.colorbar(img, ax=ax)
        plt.tight_layout()
        plt.show()

        # --- Emitir la predicción ---
        print("\n🎤 RESULTADO DE LA PRUEBA CIEGA:")
        resultado = predecir_comando(mejor_modelo, mfcc_prueba)
    else:
        print("\n❌ No se pudo procesar el audio. Verifica el formato y el archivo.")

In [ ]:
# =============================================================================
# CELDA 16 (BONUS): Predicción interactiva — Subir archivo directamente
# =============================================================================
# Esta celda permite subir un archivo de audio directamente desde tu
# computadora usando el widget de carga de Google Colab.

from google.colab import files
import io

print("📤 Sube un archivo de audio para predecir...")
print("   (Formatos aceptados: WAV, MP3, OGG, FLAC, M4A, etc.)\n")

# Abrir diálogo de subida
uploaded = files.upload()

for nombre_archivo, contenido in uploaded.items():
    # Guardar temporalmente el archivo subido
    ruta_temp = f'/content/{nombre_archivo}'
    with open(ruta_temp, 'wb') as f:
        f.write(contenido)

    print(f"\n📂 Archivo recibido: {nombre_archivo}")
    print(f"   Tamaño: {len(contenido) / 1024:.1f} KB")

    # Preprocesar y predecir
    mfcc_subido = extraer_mfcc(ruta_temp)

    if mfcc_subido is not None:
        # Visualizar MFCC
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.imshow(mfcc_subido, aspect='auto', origin='lower', cmap='magma')
        ax.set_title(f'MFCC — {nombre_archivo}', fontsize=13, fontweight='bold')
        ax.set_xlabel('Frame temporal')
        ax.set_ylabel('Coeficiente MFCC')
        plt.tight_layout()
        plt.show()

        # Predicción
        print(f"\n🎤 PREDICCIÓN para '{nombre_archivo}':")
        predecir_comando(mejor_modelo, mfcc_subido)
    else:
        print(f"   ❌ No se pudo procesar '{nombre_archivo}'.")

    # Limpiar archivo temporal
    os.remove(ruta_temp)

---

# 6. Sección de Redacción del Equipo

---

## 6.1 Justificación de la Arquitectura Elegida

> **Instrucciones para el equipo:** En esta sección, expliquen las razones técnicas detrás de la arquitectura final seleccionada. Pueden basarse en los resultados del *tuning* y la tabla comparativa.

### ¿Por qué elegimos esta arquitectura?

*[Escriban aquí la justificación. Pueden responder preguntas como:]*

- ¿Cuántas capas ocultas tiene el modelo final y por qué?
- ¿Cuántas neuronas por capa se eligieron y cómo se determinó este número?
- ¿Por qué se usó la activación ReLU en las capas ocultas?
- ¿Por qué se usó sigmoid en la capa de salida en lugar de softmax?
- ¿Qué papel jugaron BatchNormalization y Dropout en el rendimiento?
- ¿Cómo influyó el EarlyStopping en la selección del modelo?

### Comparación con otras arquitecturas probadas

*[Pueden agregar una tabla resumen o mencionar qué configuraciones funcionaron peor y por qué creen que fue así.]*

| Aspecto | Valor elegido | Alternativas probadas | Observación |
|---------|---------------|----------------------|-------------|
| Capas ocultas | *[completar]* | 1, 2, 3 | *[completar]* |
| Neuronas | *[completar]* | 32, 64, 128 | *[completar]* |
| Dropout | 0.3 | — | *[completar]* |
| Optimizador | Adam | — | *[completar]* |

---

## 6.2 Ajustes Realizados

> **Instrucciones para el equipo:** Documenten todos los ajustes que realizaron durante el desarrollo del proyecto.

### Ajustes en el Preprocesamiento

*[Describan cambios que hicieron al procesamiento de audio, por ejemplo:]*

- ¿Se modificó el número de MFCC (`N_MFCC`)? ¿A cuánto y por qué?
- ¿Se cambió la longitud máxima de padding (`MAX_PAD_LEN`)?
- ¿Se probó con diferente `SAMPLE_RATE`?
- ¿Hubo audios problemáticos que tuvieron que limpiar o re-grabar?

### Ajustes en el Modelo

*[Describan cambios en la arquitectura o el entrenamiento:]*

- ¿Se ajustó el `BATCH_SIZE`? ¿De cuánto a cuánto?
- ¿Se modificó el número de `EPOCHS` o el `PATIENCE` del EarlyStopping?
- ¿Se probaron otros optimizadores además de Adam?
- ¿Se experimentó con diferentes tasas de Dropout?

### Problemas Encontrados y Soluciones

| Problema | Solución aplicada |
|----------|------------------|
| *[Describir problema 1]* | *[Describir solución 1]* |
| *[Describir problema 2]* | *[Describir solución 2]* |
| *[Describir problema 3]* | *[Describir solución 3]* |

---

## 6.3 Conclusiones

> **Instrucciones para el equipo:** Redacten las conclusiones del proyecto abordando los siguientes puntos.

### Resultados Obtenidos

*[Resuman los resultados finales del modelo:]*

- ¿Cuál fue la accuracy final en el conjunto de prueba?
- ¿Cómo se comportó el modelo en la prueba ciega?
- ¿Hubo diferencia significativa entre la accuracy de entrenamiento y la de prueba?

### Lecciones Aprendidas

*[Reflexionen sobre lo aprendido:]*

- ¿Qué aprendieron sobre el procesamiento de audio?
- ¿Qué descubrieron sobre las redes neuronales densas?
- ¿Qué importancia tiene el preprocesamiento en un proyecto de ML?

### Trabajo Futuro

*[Mencionen posibles mejoras:]*

- ¿Se podría mejorar el modelo con más datos?
- ¿Qué otras arquitecturas se podrían explorar (CNN, RNN, LSTM)?
- ¿Cómo se podría integrar este modelo en un sistema de domótica real?
- ¿Se podrían agregar más comandos de voz al sistema?

### Reflexión Final

*[Escriban una reflexión general sobre el proyecto y su relevancia en el campo de la minería de datos y la inteligencia artificial.]*

---

## 📚 Referencias

*[Agreguen las referencias bibliográficas utilizadas, por ejemplo:]*

1. McFee, B. et al. (2015). *librosa: Audio and Music Signal Analysis in Python*. Proceedings of the 14th Python in Science Conference.
2. Chollet, F. (2021). *Deep Learning with Python* (2nd ed.). Manning Publications.
3. Documentación de TensorFlow/Keras: https://www.tensorflow.org/
4. Documentación de librosa: https://librosa.org/
5. *[Agregar más referencias según sea necesario]*

---

**Proyecto: Reconocimiento de Voz — Luz Inteligente** | Minería de Datos | Mayo 2026